## 10. PD Modeling — LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.frozen import FrozenEstimator

model_df_lgb = df_cleaned[feature_cols + [target_col]].dropna()
X_lgb = model_df_lgb[feature_cols]
y_lgb = model_df_lgb[target_col]

X_train_lgb, X_temp_lgb, y_train_lgb, y_temp_lgb = train_test_split(
    X_lgb, y_lgb, test_size=0.30, random_state=42, stratify=y_lgb)
X_val_lgb, X_test_lgb, y_val_lgb, y_test_lgb = train_test_split(
    X_temp_lgb, y_temp_lgb, test_size=0.50, random_state=42, stratify=y_temp_lgb)

print(f"Tuning LightGBM with {X_train_lgb.shape[0]:,} samples...")

param_grid = {
    'n_estimators': [300, 500], 'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6], 'num_leaves': [31, 63],
    'min_child_samples': [100, 200], 'subsample': [0.8], 'colsample_bytree': [0.8],
}
base_lgb = lgb.LGBMClassifier(
    scale_pos_weight=len(y_train_lgb[y_train_lgb==0]) / len(y_train_lgb[y_train_lgb==1]),
    random_state=42, n_jobs=-1
)
search = RandomizedSearchCV(
    base_lgb, param_distributions=param_grid, n_iter=5,
    scoring='roc_auc', cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42, n_jobs=-1, verbose=1
)
search.fit(X_train_lgb, y_train_lgb)

lgb_model = search.best_estimator_
calibrated_lgb = CalibratedClassifierCV(FrozenEstimator(lgb_model), method='sigmoid', cv='prefit')
calibrated_lgb.fit(X_val_lgb, y_val_lgb)

cal_probs_lgb = calibrated_lgb.predict_proba(X_test_lgb)[:, 1]
print(f"Final LightGBM Test AUC: {roc_auc_score(y_test_lgb, cal_probs_lgb):.4f}")